# 🔍 Exploration du MCP Pendle

Ce notebook permet de tester interactivement tous les outils du MCP Pendle.

**Endpoint MCP** : `https://api-v2.pendle.finance/core/mcp`

**Protocole** : JSON-RPC 2.0 via StreamableHTTP (SSE)

**Headers requis** :
- `Accept: application/json, text/event-stream`
- `Content-Type: application/json`

In [1]:
# Setup
import httpx
import json
from datetime import datetime, timezone

MCP_URL = "https://api-v2.pendle.finance/core/mcp"
HEADERS = {
    "Accept": "application/json, text/event-stream",
    "Content-Type": "application/json",
}

_req_id = 0

def mcp_call(method: str, params: dict = None) -> dict:
    """Make a JSON-RPC call to the Pendle MCP and parse the SSE response."""
    global _req_id
    _req_id += 1
    payload = {
        "jsonrpc": "2.0",
        "id": _req_id,
        "method": method,
    }
    if params:
        payload["params"] = params
    
    r = httpx.post(MCP_URL, headers=HEADERS, json=payload, timeout=30)
    # Parse SSE response
    text = r.text
    if "data: " in text:
        data_line = text.split("data: ", 1)[1]
        obj = json.loads(data_line)
    else:
        obj = json.loads(text)
    
    if "error" in obj:
        raise Exception(f"MCP Error: {obj['error']}")
    
    return obj["result"]


def mcp_tool(name: str, arguments: dict = None):
    """Call an MCP tool and return the parsed JSON content."""
    params = {"name": name}
    if arguments:
        params["arguments"] = arguments
    result = mcp_call("tools/call", params)
    content = result["content"][0]["text"]
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        print(f"⚠️ Response is not JSON, returning raw text ({len(content)} chars):")
        print(content[:500] if content else '(empty)')
        return content


print("✅ Helper functions ready")

✅ Helper functions ready


---
## 1. Lister tous les outils MCP disponibles

In [2]:
# Liste des 25 outils MCP
result = mcp_call("tools/list")
tools = result["tools"]
print(f"Nombre d'outils : {len(tools)}\n")
for t in tools:
    print(f"  • {t['name']} — {t['description'][:80]}...")

Nombre d'outils : 25

  • get_chains — List all blockchain network IDs that have active Pendle markets....
  • get_markets — Query Pendle markets with structured filters, sorting, and pagination. Only safe...
  • get_market — Get full details for a single Pendle market by its address and chain ID. Returns...
  • get_history — Get time-series data for a market's APY and key metrics from the in-memory marke...
  • get_external_protocols — Get external protocol integrations (Aave, Morpho, Euler, restaking) for Pendle m...
  • get_asset — Get metadata for a Pendle token (PT, YT, SY, LP) or any underlying asset by its ...
  • get_prices — Get USD prices for Pendle tokens. Returns price data from backend asset cache. O...
  • resolve_token — Resolve a token symbol or name to its address on a given chain. Searches Pendle'...
  • pendle_router — Start here when unsure which Pendle tool to use. Describes all available tools a...
  • get_portfolio — Get all Pendle positions for a wallet across a

---
## 2. `get_chains` — Chaînes actives

In [3]:
chains = mcp_tool("get_chains")
print(f"Chaînes actives : {chains}")

Chaînes actives : {'chainIds': [1, 10, 56, 146, 999, 5000, 8453, 9745, 42161, 80094]}


---
## 3. `get_markets` — Marchés Pendle

Paramètres disponibles :
- `filter` : array de `{field, op, value}` (AND-combined)
- `sort` : `{field, direction}` (asc/desc)
- `include` : champs additionnels (ou `["all"]`)
- `limit` : max 50
- `skip` : pagination

In [4]:
# Top 5 marchés Arbitrum par implied APY
markets_arb = mcp_tool("get_markets", {
    "filter": [
        {"field": "chainId", "op": "=", "value": 42161},
        {"field": "details_impliedApy", "op": ">", "value": 0.01},
    ],
    "sort": {"field": "details_impliedApy", "direction": "desc"},
    "include": ["all"],
    "limit": 5,
})

print(f"Nombre de marchés : {len(markets_arb)}\n")
for m in markets_arb:
    print(f"  {m['name']} | {m['protocol']}")
    print(f"    Implied APY: {m.get('details_impliedApy', 0)*100:.2f}%")
    print(f"    Underlying APY: {m.get('details_underlyingApy', 0)*100:.2f}%")
    print(f"    TVL: ${m.get('details_totalTvl', 0)/1e6:.1f}M")
    print(f"    Expiry: {m.get('expiry')}")
    print(f"    PT discount: {m.get('details_ptDiscount', 0)*100:.2f}%")
    print()

Nombre de marchés : 5

  sUSDai | USD.AI
    Implied APY: 8.21%
    Underlying APY: 5.59%
    TVL: $30.9M
    Expiry: 2026-10-15T00:00:00.000Z
    PT discount: 4.05%

  sUSDai | USD.AI
    Implied APY: 7.45%
    Underlying APY: 5.59%
    TVL: $13.4M
    Expiry: 2026-06-18T00:00:00.000Z
    PT discount: 1.41%

  thBILL | Theo
    Implied APY: 7.12%
    Underlying APY: 2.93%
    TVL: $5.3M
    Expiry: 2026-06-18T00:00:00.000Z
    PT discount: 1.35%

  USDai | USD.ai
    Implied APY: 5.85%
    Underlying APY: 0.00%
    TVL: $17.3M
    Expiry: 2026-10-15T00:00:00.000Z
    PT discount: 2.93%

  gUSDC | Gains Network
    Implied APY: 5.61%
    Underlying APY: 5.85%
    TVL: $0.4M
    Expiry: 2026-06-25T00:00:00.000Z
    PT discount: 1.18%



In [5]:
# Top 5 marchés Ethereum mainnet par TVL (actifs)
markets_eth = mcp_tool("get_markets", {
    "filter": [
        {"field": "chainId", "op": "=", "value": 1},
        {"field": "details_impliedApy", "op": ">", "value": 0.01},
    ],
    "sort": {"field": "details_totalTvl", "direction": "desc"},
    "include": ["all"],
    "limit": 5,
})

for m in markets_eth:
    print(f"  {m['name']} | {m['protocol']} | TVL: ${m.get('details_totalTvl', 0)/1e6:.1f}M | APY: {m.get('details_impliedApy', 0)*100:.2f}%")

  sUSDe | Ethena | TVL: $423.7M | APY: 3.75%
  USDG | Global Dollar | TVL: $104.2M | APY: 5.08%
  reUSD | re.xyz | TVL: $85.0M | APY: 8.52%
  sNUSD | Neutrl | TVL: $67.3M | APY: 8.48%
  apyUSD | APYX | TVL: $34.2M | APY: 14.83%


In [6]:
# Exploration : tous les champs d'un marché
print("Champs disponibles dans un marché :")
if markets_arb:
    for key in sorted(markets_arb[0].keys()):
        print(f"  {key}: {repr(markets_arb[0][key])[:100]}")

Champs disponibles dans un marché :
  address: '0xcbf629c8d396b1261f81f55175afa010e94787d8'
  categoryIds: '["stables","rwa","points"]'
  chainId: 42161
  details_aggregatedApy: 0.07054479572874006
  details_impliedApy: 0.08206737181558932
  details_liquidity: 13430400.287426708
  details_lpRewardApy: 0
  details_maxBoostedApy: 0.08521827790645298
  details_pendleApy: 0.009782321451808615
  details_ptDiscount: 0.040504578273711456
  details_swapFeeApy: 0.0007533695393395767
  details_totalActiveSupply: 5595324.940765481
  details_totalPt: 2198487.9940743535
  details_totalSy: 10504100.668694071
  details_totalTvl: 30880983.551507764
  details_tradingVolume: 114803.82348277855
  details_underlyingApy: 0.0558984171277781
  details_voterApy: 0
  details_ytFloatingApy: -0.5229761531835744
  expiry: '2026-10-15T00:00:00.000Z'
  isPrime: 1
  name: 'sUSDai'
  protocol: 'USD.AI'
  pt: '42161-0xb459db106f645d698e74027eef6019a26a0675cc'
  sy: '42161-0x30ccf4bbee313fcd19f3e295b3ba2920a24e2f62'
  

---
## 4. `get_market` — Détail complet d'un marché

In [7]:
# Prendre le premier marché Arbitrum et obtenir son détail complet
if markets_arb:
    sample = markets_arb[0]
    market_detail = mcp_tool("get_market", {
        "chainId": sample["chainId"],
        "market": sample["address"],
    })
    print(json.dumps(market_detail, indent=2))

{
  "identity": {
    "address": "0xcbf629c8d396b1261f81f55175afa010e94787d8",
    "chainId": 42161,
    "name": "sUSDai",
    "symbol": "PENDLE-LPT",
    "expiry": "2026-10-15T00:00:00.000Z",
    "protocol": "USD.AI",
    "isWhitelisted": true,
    "isWhitelistedPro": true,
    "status": 1,
    "categoryIds": [
      "stables",
      "rwa",
      "points"
    ]
  },
  "tokens": {
    "pt": "42161-0xb459db106f645d698e74027eef6019a26a0675cc",
    "yt": "42161-0x11456849c38ea4af212ab8d4324b39983716516a",
    "sy": "42161-0x30ccf4bbee313fcd19f3e295b3ba2920a24e2f62",
    "underlyingAsset": "42161-0x0b2b2b2076d95dda7817e785989fe353fe955ef9",
    "accountingAsset": "42161-0x0a1a1a107e45b7ced86833863f482bc5f4ed82ef",
    "rewardTokens": [
      "42161-0x0c880f6761f1af8d9aa9c466984b80dab9a8c9e8"
    ],
    "inputTokens": [
      "42161-0x0a1a1a107e45b7ced86833863f482bc5f4ed82ef",
      "42161-0x0b2b2b2076d95dda7817e785989fe353fe955ef9",
      "42161-0x46850ad61c2b7d64d08c9c754f45254596696984"


---
## 5. `get_external_protocols` — Intégrations lending par marché

**C'est la donnée clé pour les loops !**

Un marché avec des money markets (Morpho, Euler, Aave, Silo…) dans ses external protocols = **loop possible** (PT comme collateral → borrow underlying → re-buy PT).

In [9]:
# Protocoles externes pour Ethereum (chainId requis)
all_protocols = mcp_tool("get_external_protocols", {"chainId": 1})
if isinstance(all_protocols, dict):
    print(f"Nombre total de protocoles : {all_protocols.get('total', len(all_protocols.get('protocols', [])))}\n")
    for p in all_protocols.get("protocols", []):
        print(f"  [{p.get('category')}] {p.get('name')} ({p.get('id')})")
else:
    print(all_protocols)

Nombre total de protocoles : 35

  [yield strategy] Contango (contango)
  [yield strategy] yearn (yearn)
  [money market] Venus Protocol (venus)
  [money market] Silo (silo)
  [money market] Morpho (morpho)
  [money market] Euler (euler)
  [money market] Gearbox (gearbox)
  [money market] Aave (aave)
  [money market] TermMax (termmax)
  [yield strategy] YieldFi (yield-fi)
  [cex / web3 wallet] Binance (binance)
  [cex / web3 wallet] Bybit (bybit)
  [cex / web3 wallet] Bitget (bitget)
  [cex / web3 wallet] Coinbase (coinbase)
  [money market] Dolomite (dolomite)
  [cex / web3 wallet] Gate.io (gateio)
  [liquid locker] Penpie (penpie)
  [liquid locker] Equilibria (equilibria)
  [insurance] InsurACE (insurace)
  [money market] Spark (spark)
  [cex / web3 wallet] OKX (okx)
  [money market] Lista Lending (lista)
  [cex / web3 wallet] Upbit (upbit)
  [liquid locker] StakeDAO (stakedao)
  [money market] FiRM (firm)
  [money market] HyperLend (hyperlend)
  [yield strategy] DeFi Saver (defisave

In [10]:
# Protocoles externes pour un marché SPÉCIFIQUE (Ethereum sUSDe)
# Ce marché a beaucoup d'intégrations
proto_susde = mcp_tool("get_external_protocols", {
    "chainId": 1,
    "market": "0x8dae8ece668cf80d348873f23d456448e8694883",
})

print(f"Marché: {proto_susde.get('market')}")
print(f"Protocoles: {len(proto_susde.get('protocols', []))}\n")
for p in proto_susde.get("protocols", []):
    print(f"  [{p.get('category')}] {p.get('name')}")
    print(f"    → {p.get('description', '')[:100]}")
    print(f"    URL: {p.get('url', '')}")
    print()

Marché: 1-0x8dae8ece668cf80d348873f23d456448e8694883
Protocoles: 5

  [yield strategy] Contango
    → Contango builds positions by automating looping strategies through flash loans. When a trader opens 
    URL: https://app.contango.xyz/

  [money market] Morpho
    → A permissionless and non-custodial lending protocol where users can earn interest on over-collateriz
    URL: https://app.morpho.org/ethereum/borrow

  [money market] Euler
    → Euler is a modular lending platform that enables users to lend, borrow and build without limits.
    URL: https://app.euler.finance/borrow?network=ethereum

  [money market] Aave
    → Aave is a decentralised non-custodial liquidity protocol where users can participate as suppliers or
    URL: https://app.aave.com/

  [money market] TermMax
    → TermMax is a DeFi AMM protocol that allows users to lend, borrow and one-click leverage at fixed-rat
    URL: https://app.termmax.ts.finance/borrow?chain=eth



In [ ]:
# Tester un marché sans intégrations
if markets_arb:
    for m in markets_arb:
        proto = mcp_tool("get_external_protocols", {
            "chainId": m["chainId"],
            "market": m["address"],
        })
        protos = proto.get("protocols", [])
        mm_count = sum(1 for p in protos if p.get("category", "").lower() == "money market")
        has_contango = any(p.get("id") == "contango" for p in protos)
        print(f"  {m['name']} | protocols: {len(protos)} | money markets: {mm_count} | contango: {has_contango}")

---
## 6. `get_asset` — Metadata d'un token

In [ ]:
# Metadata d'un token (ex: underlying asset du premier marché)
if markets_arb:
    underlying = markets_arb[0].get("underlyingAsset", "")
    # Format: "chainId-address"
    if "-" in underlying:
        chain_id, addr = underlying.split("-", 1)
        asset = mcp_tool("get_asset", {
            "chainId": int(chain_id),
            "address": addr,
        })
        print(json.dumps(asset, indent=2))

---
## 7. `get_prices` — Prix USD des tokens

In [ ]:
# Prix de tous les tokens sur Ethereum
prices = mcp_tool("get_prices", {"chainId": 1})
print(f"Nombre de prix : {len(prices) if isinstance(prices, (list, dict)) else 'N/A'}")
# Afficher les 10 premiers
if isinstance(prices, dict):
    for addr, price in list(prices.items())[:10]:
        print(f"  {addr}: ${price}")
elif isinstance(prices, list):
    for p in prices[:10]:
        print(f"  {p}")

---
## 8. `get_history` — Données historiques APY

In [ ]:
# Historique APY d'un marché
if markets_arb:
    sample = markets_arb[0]
    history = mcp_tool("get_history", {
        "chainId": sample["chainId"],
        "market": sample["address"],
        "fields": "timestamp,impliedApy,underlyingApy",
    })
    if isinstance(history, list):
        print(f"Points de données : {len(history)}")
        print(f"Dernier : {history[-1] if history else 'N/A'}")
        print(f"Premier : {history[0] if history else 'N/A'}")
    else:
        print(json.dumps(history, indent=2)[:500])

---
## 9. `resolve_token` — Résoudre un symbole en adresse

In [ ]:
# Résoudre "USDC" sur Ethereum
usdc = mcp_tool("resolve_token", {
    "chainId": 1,
    "query": "USDC",
})
print(json.dumps(usdc, indent=2))

In [ ]:
# Résoudre "wstETH" sur Arbitrum
wsteth = mcp_tool("resolve_token", {
    "chainId": 42161,
    "query": "wstETH",
})
print(json.dumps(wsteth, indent=2))

---
## 10. `pendle_router` — Aide au routage

In [ ]:
router = mcp_tool("pendle_router", {
    "intent": "find the best looping opportunities with fixed yield",
})
print(json.dumps(router, indent=2))

---
## 11. Exploration complète : marchés + protocoles (pour comprendre les loops)

Ici on combine `get_markets` + `get_external_protocols` pour chaque marché, afin de voir lesquels sont loopables.

In [ ]:
# Récupérer top 10 marchés Ethereum par APY, puis checker les protocols de chacun
markets = mcp_tool("get_markets", {
    "filter": [
        {"field": "chainId", "op": "=", "value": 1},
        {"field": "details_impliedApy", "op": ">", "value": 0.01},
    ],
    "sort": {"field": "details_impliedApy", "direction": "desc"},
    "include": ["all"],
    "limit": 10,
})

print(f"Marchés récupérés : {len(markets)}\n")
print("=" * 100)

for m in markets:
    # Get external protocols
    proto = mcp_tool("get_external_protocols", {
        "chainId": m["chainId"],
        "market": m["address"],
    })
    protocols = proto.get("protocols", [])
    
    money_markets = [p for p in protocols if p.get("category", "").lower() == "money market"]
    yield_strats = [p for p in protocols if p.get("category", "").lower() == "yield strategy"]
    has_contango = any(p.get("id") == "contango" for p in protocols)
    
    implied = m.get("details_impliedApy", 0) * 100
    underlying = m.get("details_underlyingApy", 0) * 100
    spread = implied - underlying
    
    loopable = "✅ LOOPABLE" if (money_markets or has_contango) else "❌ pas de lending"
    
    print(f"\n{m['name']} | {m.get('protocol', '')} | {loopable}")
    print(f"  Implied APY: {implied:.2f}% | Underlying: {underlying:.2f}% | Spread: {spread:.2f}%")
    print(f"  TVL: ${m.get('details_totalTvl', 0)/1e6:.1f}M | Expiry: {m.get('expiry', '?')}")
    
    if money_markets:
        print(f"  🏦 Money Markets: {', '.join(p.get('name', '') for p in money_markets)}")
    if has_contango:
        print(f"  ⚡ Contango (automated looping via flash loans)")
    if yield_strats:
        print(f"  📈 Yield Strategies: {', '.join(p.get('name', '') for p in yield_strats)}")

print("\n" + "=" * 100)

---
## 12. `preview_trade` — Simuler un trade (read-only)

In [ ]:
# Preview: acheter du PT avec 1000 USDC sur un marché Ethereum
# (Nécessite une adresse receiver mais pas de wallet)
# Décommentez pour tester :

# preview = mcp_tool("preview_trade", {
#     "chainId": 1,
#     "action": "buy_pt",
#     "market": "0x8dae8ece668cf80d348873f23d456448e8694883",
#     "tokenIn": "0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48",  # USDC
#     "humanAmount": 1000,
#     "receiver": "0x0000000000000000000000000000000000000001",
# })
# print(json.dumps(preview, indent=2))

print("Décommentez le code ci-dessus pour tester preview_trade")

---
## 13. Sandbox — Cellule libre pour tester

In [ ]:
# Votre test ici :
# result = mcp_tool("nom_de_l_outil", {"param": "valeur"})
# print(json.dumps(result, indent=2))